**Importa Bibliotecas principais**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**Import do dataset**

In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
)

C:\Users\Guima\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Guima\AppData\Local\Temp\ipykernel_31564\1404267617.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


**Tratativa de coluna Object para float**

In [4]:
#Ajustes de padronizão de TotalCharges de objet to float
df[df["TotalCharges"].str.strip() == ""]
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')

**Tratamento de missing values por mediana**

In [5]:
#Trata NaN populando pela média
mediana = df["TotalCharges"].median()
df["TotalCharges"] = df["TotalCharges"].fillna(mediana)

**Distribuição da variavél target**

In [6]:
# Mapear target para binário
df.rename(columns={'Churn': 'target'}, inplace=True)
df['target'] = df['target'].map({'No': 0, 'Yes': 1})

print(f"Distribuição target:\n{df['target'].value_counts(normalize=True)}")

Distribuição target:
target
0    0.73463
1    0.26537
Name: proportion, dtype: float64


In [7]:
!pip install mlflow

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import mlflow
print(mlflow.__version__)

3.10.1


In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    average_precision_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

# --- Seed para reprodutibilidade (obrigatório no Tech Challenge) ---
SEED = 42
np.random.seed(SEED)

# --- Separar features e target ---
X = df.drop(columns=['customerID', 'target'])
y = df['target']

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"Distribuição target:\n{y.value_counts(normalize=True)}")

Shape X: (7043, 19)
Shape y: (7043,)
Distribuição target:
target
0    0.73463
1    0.26537
Name: proportion, dtype: float64


In [10]:
# Identificar colunas numéricas e categóricas
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

# Pipeline de pré-processamento
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

print("Preprocessor configurado.")
print(f"  - Numéricas ({len(numeric_features)}): {numeric_features}")
print(f"  - Categóricas ({len(categorical_features)}): {categorical_features}")

Preprocessor configurado.
  - Numéricas (3): ['tenure', 'MonthlyCharges', 'TotalCharges']
  - Categóricas (16): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [11]:
# --- Definição dos modelos baseline ---
baselines = {
    'DummyClassifier (stratified)': DummyClassifier(strategy='stratified', random_state=SEED),
    'DummyClassifier (most_frequent)': DummyClassifier(strategy='most_frequent', random_state=SEED),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
}

# --- Validação cruzada estratificada ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

results = []

for model_name, model in baselines.items():
    fold_metrics = {'auc_roc': [], 'pr_auc': [], 'f1': [], 'recall': [], 'precision': []}
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # Criar pipeline completo (preprocessor + modelo)
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
        
        pipe.fit(X_train, y_train)
        
        # Predições
        y_pred = pipe.predict(X_val)
        y_proba = pipe.predict_proba(X_val)[:, 1]
        
        # Métricas
        fold_metrics['auc_roc'].append(roc_auc_score(y_val, y_proba))
        fold_metrics['pr_auc'].append(average_precision_score(y_val, y_proba))
        fold_metrics['f1'].append(f1_score(y_val, y_pred))
        fold_metrics['recall'].append(recall_score(y_val, y_pred))
        fold_metrics['precision'].append(precision_score(y_val, y_pred))
    
    # Média e desvio padrão de cada métrica
    row = {'model': model_name}
    for metric, values in fold_metrics.items():
        row[f'{metric}_mean'] = np.mean(values)
        row[f'{metric}_std'] = np.std(values)
    results.append(row)
    
    print(f"\n✅ {model_name}")
    print(f"   AUC-ROC: {row['auc_roc_mean']:.4f} ± {row['auc_roc_std']:.4f}")
    print(f"   PR-AUC:  {row['pr_auc_mean']:.4f} ± {row['pr_auc_std']:.4f}")
    print(f"   F1:      {row['f1_mean']:.4f} ± {row['f1_std']:.4f}")
    print(f"   Recall:  {row['recall_mean']:.4f} ± {row['recall_std']:.4f}")

results_df = pd.DataFrame(results)
results_df


✅ DummyClassifier (stratified)
   AUC-ROC: 0.5050 ± 0.0073
   PR-AUC:  0.2675 ± 0.0031
   F1:      0.2738 ± 0.0107
   Recall:  0.2750 ± 0.0107

✅ DummyClassifier (most_frequent)
   AUC-ROC: 0.5000 ± 0.0000
   PR-AUC:  0.2654 ± 0.0002
   F1:      0.0000 ± 0.0000
   Recall:  0.0000 ± 0.0000

✅ Logistic Regression
   AUC-ROC: 0.8449 ± 0.0135
   PR-AUC:  0.6565 ± 0.0267
   F1:      0.6015 ± 0.0278
   Recall:  0.5532 ± 0.0318


,model,auc_roc_mean,auc_roc_std,pr_auc_mean,pr_auc_std,f1_mean,f1_std,recall_mean,recall_std,precision_mean,precision_std
0,DummyClassifier (stratified),0.505013,0.007284,0.267493,0.003128,0.273836,0.010724,0.275004,0.010652,0.272679,0.010794
1,DummyClassifier (most_frequent),0.500000,0.000000,0.265370,0.000239,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Logistic Regression,0.844917,0.013538,0.656502,0.026669,0.601529,0.027801,0.553211,0.031773,0.659382,0.022406


## Interpretação dos Baselines

- **DummyClassifier (stratified)**: referência mínima — qualquer modelo real precisa superar isso.
- **DummyClassifier (most_frequent)**: sempre prevê "não churn" — AUC-ROC ~0.50, Recall = 0.
- **Logistic Regression**: baseline inteligente — já captura padrões lineares dos dados.

A Regressão Logística define o **piso de performance** que a MLP (Etapa 2) precisa superar.